# 00 - Scaffold tour

Milestone 0 of `vol-strategy-backtester`. No market data yet: this notebook tours the repo layout, the Python-to-app JSON contract, and the one piece of real code so far, an **illustrative** SSVI implied-volatility surface.

This notebook is generated by `tools/build_notebooks.py`. Do not edit it by hand; edit `src/` and rebuild.

## Repo layout

```text
src/volbt/        the library (the single source of truth for code)
  config.py         where raw data lives (outside the repo)
  contract.py       the JSON envelope Python writes for the app
  illustrative.py   the illustrative SSVI surface (this notebook)
tests/            pytest; expected values worked out by hand
tools/            build_notebooks.py, export_app_data.py
notebooks/        generated from src/ by tools/build_notebooks.py
app/              the React app; reads only app/public/data/*.json
docs/             ADRs, architecture notes, concept notes
```

Raw vendor data (Kaggle, DoltHub) is big and licensed, so it lives outside the repo, in `~/data/vol-projects` by default (override with `VOL_DATA_DIR`). It is never committed.

## The Python-to-app contract

Python writes JSON into `app/public/data/`; the React app reads only those files. Nothing else crosses the boundary. Every file starts with the same envelope:

| field | meaning |
|---|---|
| `schema_version` | layout version of the file, currently `0.3.0` |
| `generated_at` | UTC time written, ISO-8601 ending in `Z` |
| `source_commit` | `git rev-parse HEAD`, plus `-dirty` if the tree has uncommitted changes, or `uncommitted` if there is no commit yet |

`manifest.json` lists the datasets; `illustrative_surface.json` holds the surface as `moneyness` (log-moneyness k), `dte` (days to expiry) and `iv[i_dte][j_k]` (decimal vol).

## The illustrative SSVI surface

**Illustrative shape, not market data** - replaced by surfaces fitted to real SPY chains in milestone 2 (the pricing core).

SSVI (Gatheral & Jacquier, 2014) writes total variance `w = sigma^2 * T` as

`w(k, theta) = theta/2 * (1 + rho*phi*k + sqrt((phi*k + rho)^2 + 1 - rho^2))`

with `k = ln(K/F)`, `phi = eta / sqrt(theta)` and `theta = sigma_atm(T)^2 * T`. The cells below are copied from `src/volbt/illustrative.py`.

In [ ]:
import numpy as np

In [ ]:
# ATM vol goes in a straight line from 16% at 7 days to 20% at 365 days. Equity
# index vol is normally a little higher further out; the exact numbers are a
# choice made for looks.
SIGMA_ATM_SHORT = 0.16
SIGMA_ATM_LONG = 0.20
# Skew and curvature. rho = -0.7 is a typical equity-index skew; eta = 1.0 keeps
# the smile well inside the no-arbitrage limits (see no_arbitrage_checks).
RHO = -0.7
ETA = 1.0
# The grid the app draws: 20 expiries from 7 to 365 days, 26 log-moneyness points
# from -0.30 to +0.20 (steps of 0.02, so k = 0 is exactly on the grid).
DTE_MIN, DTE_MAX, N_DTE = 7, 365, 20
K_MIN, K_MAX, N_K = -0.30, 0.20, 26
DAYS_PER_YEAR = 365
# The log-moneyness points where the calendar check compares expiries. Wider than the
# drawn grid (-0.30..0.20) on purpose: arbitrage hides in the wings, not where we draw.
# Rounded for the same reason as the drawn grid (linspace float noise).
CALENDAR_CHECK_K = np.round(np.linspace(-1.0, 1.0, 201), 10)

In [ ]:
def sigma_atm(dte, sigma_short=SIGMA_ATM_SHORT, sigma_long=SIGMA_ATM_LONG):
    """ATM vol at `dte` days: a straight line from `sigma_short` (at DTE_MIN) to `sigma_long` (at DTE_MAX)."""
    fraction = (np.asarray(dte, dtype=float) - DTE_MIN) / (DTE_MAX - DTE_MIN)
    return sigma_short + (sigma_long - sigma_short) * fraction

In [ ]:
def atm_total_variance(dte, sigma_short=SIGMA_ATM_SHORT, sigma_long=SIGMA_ATM_LONG):
    """theta(T) = sigma_atm(T)^2 * T, with T = dte / 365 years."""
    years = np.asarray(dte, dtype=float) / DAYS_PER_YEAR
    return sigma_atm(dte, sigma_short, sigma_long) ** 2 * years

In [ ]:
def phi(theta, eta=ETA):
    """How fast the smile bends away from the money: phi(theta) = eta / sqrt(theta)."""
    return eta / np.sqrt(theta)

In [ ]:
def total_variance(k, theta, rho=RHO, eta=ETA):
    """SSVI total variance w(k, theta) = theta/2 * (1 + rho*phi*k + sqrt((phi*k + rho)^2 + 1 - rho^2))."""
    phi_k = phi(theta, eta) * k
    # At k = 0 this collapses to theta/2 * (1 + sqrt(rho^2 + 1 - rho^2)) = theta,
    # which is why theta is "the ATM total variance".
    return theta / 2 * (1 + rho * phi_k + np.sqrt((phi_k + rho) ** 2 + 1 - rho**2))

In [ ]:
def iv_grid(rho=RHO, eta=ETA, sigma_short=SIGMA_ATM_SHORT, sigma_long=SIGMA_ATM_LONG):
    """Return (k_grid, dte_grid, iv) with iv[i_dte][j_k] as a decimal (0.20 = 20%)."""
    # Round away float noise: linspace gives -0.27999999999999997 where we mean -0.28.
    k_grid = np.round(np.linspace(K_MIN, K_MAX, N_K), 10)
    # Whole days: the app labels expiries as "30d", not "30.4d".
    dte_grid = np.round(np.linspace(DTE_MIN, DTE_MAX, N_DTE)).astype(int)
    # Column vectors (shape (N_DTE, 1)) so numpy broadcasts them against the k
    # row (shape (N_K,)) into a full (N_DTE, N_K) table: one row per expiry.
    theta = atm_total_variance(dte_grid, sigma_short, sigma_long)[:, None]
    years = (dte_grid / DAYS_PER_YEAR)[:, None]
    w = total_variance(k_grid, theta, rho, eta)
    # w = sigma^2 * T, so sigma = sqrt(w / T).
    return k_grid, dte_grid, np.sqrt(w / years)

In [ ]:
def butterfly_conditions(theta, rho=RHO, eta=ETA):
    """The two SSVI butterfly conditions, as a pure function of the arrays `theta`, `rho`, `eta`.

    butterfly_1: theta*phi*(1+|rho|) < 4      (strict)
    butterfly_2: theta*phi^2*(1+|rho|) <= 4   (not strict)
        Together they rule out negative option-price curvature in strike, i.e.
        a butterfly spread (long wings, short the middle) with negative cost.
        Gatheral & Jacquier, Theorem 4.2. Each condition must hold at EVERY theta
        passed in; the one expiry that breaks it fails the whole surface.

    Kept free of the term structure and the grid so a test can feed it exact values
    and probe each boundary (equal to 4, just above, just below).

    The theorem asks for the conditions at EVERY theta > 0, but we only see the thetas passed
    in. For phi = eta/sqrt(theta), theta*phi*(1+|rho|) = eta*sqrt(theta)*(1+|rho|) keeps growing
    with theta, so the surface can be certified only up to a maximum expiry (Gatheral &
    Jacquier, Remark 4.4: t*_0, where that product reaches 4). The shipped surface ends far
    short of it.
    """
    theta = np.asarray(theta, dtype=float)
    phi_theta = phi(theta, eta)
    one_plus_abs_rho = 1 + abs(rho)
    return {
        "butterfly_1": bool(np.all(theta * phi_theta * one_plus_abs_rho < 4)),
        "butterfly_2": bool(np.all(theta * phi_theta**2 * one_plus_abs_rho <= 4)),
    }

In [ ]:
def calendar_condition(theta, w):
    """No calendar-spread arbitrage on a total-variance table: nothing shrinks as expiry grows.

    `theta` is the ATM total variance per expiry (ascending expiry) and `w[i][j]` the total
    variance at expiry i and log-moneyness j. A longer-dated option cannot be cheaper than a
    shorter-dated one at the same strike, which in w means w(k, T) never decreases in T at
    ANY k, not just at the money. Equal values are allowed.
    """
    theta = np.asarray(theta, dtype=float)
    w = np.asarray(w, dtype=float)
    return bool(np.all(np.diff(theta) >= 0) and np.all(np.diff(w, axis=0) >= 0))

In [ ]:
def no_arbitrage_checks(
    dte,
    rho=RHO,
    eta=ETA,
    sigma_short=SIGMA_ATM_SHORT,
    sigma_long=SIGMA_ATM_LONG,
    k=CALENDAR_CHECK_K,
):
    """Check the SSVI no-arbitrage conditions over the expiries in `dte` (ascending).

    butterfly_1, butterfly_2: see `butterfly_conditions`.
    calendar: `calendar_condition` on the total variance w(k, T) actually computed for these
        expiries over the log-moneyness points `k`, plus theta(T) non-decreasing. The ATM term
        structure (`sigma_short` -> `sigma_long`) is a parameter, so an inverted term structure
        (short-dated vol far above long-dated, theta falling with T) really does fail.

    What this does and does not prove about calendar arbitrage. Gatheral & Jacquier,
    Theorem 4.1: SSVI is free of calendar-spread arbitrage if and only if theta is
    non-decreasing in T (checked here) and 0 <= d(theta*phi)/d(theta) <= (1 + sqrt(1 - rho^2)) / rho^2 * phi
    for all theta > 0 (the upper bound is infinite when rho = 0).
    For phi = eta / sqrt(theta) we have theta*phi = eta*sqrt(theta), so
    d(theta*phi)/d(theta) = eta / (2*sqrt(theta)) = phi/2, and the condition reads
        0 <= phi/2 <= (1 + sqrt(1 - rho^2)) / rho^2 * phi.
    That holds for every theta, eta and |rho| < 1, because
    rho^2/2 <= 1/2 < 1 <= 1 + sqrt(1 - rho^2).
    So for this phi the theorem reduces to "theta non-decreasing"; the w-grid comparison is
    a second, direct check on the numbers we actually draw. It samples finitely many k, so
    it can miss a violation between the points; the analytic argument covers every k.
    """
    theta = np.atleast_1d(atm_total_variance(dte, sigma_short, sigma_long))
    # (n_expiries, n_k) table: expiries down the rows, strikes across the columns.
    w = total_variance(np.asarray(k, dtype=float)[None, :], theta[:, None], rho, eta)
    return {
        **butterfly_conditions(theta, rho, eta),
        "calendar": calendar_condition(theta, w),
    }

## Build the grid

One row per expiry, one column per log-moneyness.

In [ ]:
k_grid, dte_grid, iv = iv_grid()
print('iv shape (dte, k):', iv.shape)
print('k from', k_grid[0], 'to', k_grid[-1])
print('dte from', dte_grid[0], 'to', dte_grid[-1])

## Check the shape

At `k = 0` the surface must return the ATM vol exactly, and with `rho < 0` the low-strike side (`k < 0`) must sit above the high-strike side.

In [ ]:
atm = int(np.argmin(np.abs(k_grid)))
print('ATM column equals sigma_atm:', np.allclose(iv[:, atm], sigma_atm(dte_grid)))
print('k = -0.10 above k = +0.10 at every expiry:',
      bool(np.all(iv[:, atm - 5] > iv[:, atm + 5])))

## No-arbitrage checks

The shipped parameters pass. `eta = 2.0` is too curved: `theta*phi^2*(1+|rho|) = 4 * 1.7 = 6.8 > 4`, so `butterfly_2` fails.

The calendar check compares total variance across expiries at every strike, so it can fail too: an inverted term structure (100% vol at 7 days, 5% at one year) makes `theta` fall from 0.0192 to 0.0025 as the expiry grows.

In [ ]:
print('shipped:  ', no_arbitrage_checks(dte_grid))
print('eta = 2.0:', no_arbitrage_checks(dte_grid, eta=2.0))
print('inverted: ', no_arbitrage_checks(dte_grid, sigma_short=1.0, sigma_long=0.05))